# Paper PoRT Prefix Compiler Source Diagnostic

This notebook follows notebook 22. It isolates the prefix compiler/artifact gap by comparing raw generation against generated answers from one or more T5 prefix compiler sources on the same selected WMDP rows.

It does not bootstrap or train recreated artifacts. The default source is `google/flan-t5-small`. If a recreated artifact dir or zip is available through env vars or the usual working paths, it is added as a second source; otherwise it is recorded as skipped instead of failing.

In [ ]:
from pathlib import Path
import importlib
import json
import os
import subprocess
import sys

REPO_URL = 'https://github.com/toanthangO20/PoRT_LLM_Unlearning-Experiment.git'
REPO_DIR_NAME = 'PoRT_LLM_Unlearning-Experiment'
IS_KAGGLE = Path('/kaggle/working').exists()


def has_project_layout(path):
    path = Path(path)
    return (path / 'PoRT_pipeline' / 'WMDP' / 'port_pipeline_wmdp.py').exists() and (path / 'dataset').exists()


def clone_or_use_project():
    if IS_KAGGLE:
        target = Path('/kaggle/working') / REPO_DIR_NAME
        if has_project_layout(target):
            print(f'Using existing cloned repository: {target}')
            subprocess.check_call(['git', '-C', str(target), 'pull', '--ff-only'])
            return target.resolve()
        if target.exists():
            raise RuntimeError(f'{target} exists but does not look like this repo.')
        print(f'Cloning {REPO_URL} into {target}')
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
        return target.resolve()

    local_root = Path.cwd().resolve()
    if has_project_layout(local_root):
        return local_root
    target = local_root / REPO_DIR_NAME
    if has_project_layout(target):
        return target.resolve()
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
    return target.resolve()


PROJECT_ROOT = clone_or_use_project()
os.environ['PORT_PROJECT_ROOT'] = str(PROJECT_ROOT)
commit_sha = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
print(f'Project root: {PROJECT_ROOT}')
print(f'Commit: {commit_sha}')


In [ ]:
required_packages = {
    'datasets': 'datasets>=2.10.1',
    'joblib': 'joblib',
    'numpy': 'numpy',
    'pandas': 'pandas',
    'pyarrow': 'pyarrow>=10',
    'safetensors': 'safetensors',
    'sklearn': 'scikit-learn',
    'torch': 'torch',
    'transformers': 'transformers>=4.38.0',
    'sentencepiece': 'sentencepiece',
    'yaml': 'pyyaml',
    'tqdm': 'tqdm',
}

missing_packages = []
for module_name, package_spec in required_packages.items():
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        missing_packages.append(package_spec)

if missing_packages:
    print('Installing missing packages:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
else:
    print('Required packages are already available.')


## Runtime Config

Defaults:

- `PORT_MAX_SAMPLES=32`
- `PORT_PREFIX_INCLUDE_BASE_T5=true`
- `PORT_PREFIX_BASE_T5_MODEL_PATH=google/flan-t5-small`
- `PORT_PREFIX_INCLUDE_RECREATED_T5=true`
- `PORT_RESUME_EXISTING=true`
- `PORT_FAIL_FAST=true`

Optional extra T5 sources can be supplied as `PORT_PREFIX_EXTRA_T5_MODELS`, using `label=model_path` entries separated by `;` or `,`.

In [ ]:
os.environ.setdefault('PORT_ARTIFACT_MODE', 'recreated')
os.environ.setdefault('PORT_MAX_SAMPLES', '32')
os.environ.setdefault('PORT_PREFIX_INCLUDE_BASE_T5', 'true')
os.environ.setdefault('PORT_PREFIX_BASE_T5_MODEL_PATH', 'google/flan-t5-small')
os.environ.setdefault('PORT_PREFIX_INCLUDE_RECREATED_T5', 'true')
os.environ.setdefault('PORT_RESUME_EXISTING', 'true')
os.environ.setdefault('PORT_FAIL_FAST', 'true')
os.environ.setdefault('PORT_BEST_CLASSIFIER_SAMPLES_PER_DOMAIN', '256')
os.environ.setdefault('PORT_BEST_CLASSIFIER_WRONG_ANSWERS_PER_QUESTION', '3')
os.environ.setdefault('PORT_BEST_CLASSIFIER_FEATURE_SET', 'answer_only')
os.environ.setdefault('PORT_BEST_CLASSIFIER_MAX_FEATURES', '50000')

runtime_keys = [
    'PORT_ARTIFACT_MODE',
    'PORT_RUN_NAME',
    'PORT_WMDP_VARIANTS',
    'PORT_WMDP_DOMAINS',
    'PORT_MAX_SAMPLES',
    'PORT_PREFIX_INCLUDE_BASE_T5',
    'PORT_PREFIX_BASE_T5_MODEL_PATH',
    'PORT_PREFIX_INCLUDE_RECREATED_T5',
    'PORT_PREFIX_EXTRA_T5_MODELS',
    'PORT_RESUME_EXISTING',
    'PORT_FAIL_FAST',
    'PORT_RECREATED_ARTIFACT_DIR',
    'PORT_RECREATED_ARTIFACT_ZIP_URL',
    'PORT_RECREATED_ARTIFACT_ZIP_PATH',
]
print(json.dumps({key: os.environ.get(key) for key in runtime_keys}, indent=2))


In [ ]:
import gc
import importlib.util

gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print('Cleared CUDA cache before run.')
except Exception as exc:
    print('CUDA cache cleanup skipped:', exc)

runner_path = PROJECT_ROOT / 'notebooks' / 'common' / 'port_prefix_compiler_source_diagnostic.py'
if not runner_path.exists():
    raise FileNotFoundError(runner_path)

common_dir = str(runner_path.parent)
if common_dir not in sys.path:
    sys.path.insert(0, common_dir)

spec = importlib.util.spec_from_file_location('port_prefix_compiler_source_diagnostic', runner_path)
port_prefix_compiler_source_diagnostic = importlib.util.module_from_spec(spec)
spec.loader.exec_module(port_prefix_compiler_source_diagnostic)

result = port_prefix_compiler_source_diagnostic.run(
    project_root=PROJECT_ROOT,
    is_kaggle=IS_KAGGLE,
    commit_sha=commit_sha,
)
print(json.dumps(result, indent=2, default=str))

run_dir = Path(result['run_dir'])
for artifact_name in [
    'artifact_audit.json',
    'run_config.json',
    'summary.json',
    'all_prefix_compiler_predictions.csv',
    'prefix_compiler_summary_by_job.csv',
    'prefix_compiler_summary_overall.csv',
    'failed_jobs.json',
]:
    artifact_path = run_dir / artifact_name
    print(f'{artifact_name}: {artifact_path.exists()} {artifact_path}')
